# Forget-MI LoKU — Machine Unlearning Pipeline

> ✅ **exp10 đã sẵn sàng — không cần sửa tay gì.**
> Đã `git push` code+config mới từ local? → Trên Colab chỉ cần:
> **Cell 1** (pull code) → *(bỏ qua Cell 2, 3 nếu đã có data)* → **Cell 3.5 → Cell 4 → Cell 5**.
> Hoặc bấm **Runtime ▸ Run all** (Cell 2/3 idempotent, an toàn chạy lại).
> `EXP_NAME`/`HYPOTHESIS` của exp10 đã điền sẵn trong Cell 4.

Notebook thực hiện toàn bộ pipeline:

1. **Cell 1** — Mount Drive + pull code mới (KHÔNG xóa data đã extract)
2. **Cell 2** — Extract data & models (chỉ chạy LẦN ĐẦU)
3. **Cell 3** — Preprocess (chỉ chạy LẦN ĐẦU)
4. **Cell 3.5** — Verify config (mỗi lần chạy exp mới)
5. **Cell 4** — Huấn luyện LoKU Unlearning (với **auto-tracking**)
6. **Cell 5** — **Auto-commit & push** kết quả lên GitHub

## Workflow cho mỗi experiment mới

### Lần đầu chạy (full setup)

1. Sửa `config.yaml` ở local → push lên GitHub
2. Mở Colab → chạy **Cell 1 → Cell 2 → Cell 3** (lần đầu, đợi ~3 phút)
3. **Sửa `EXP_NAME` + `HYPOTHESIS`** ở đầu Cell 4 *(exp10 đã điền sẵn)*
4. Chạy **Cell 3.5 → Cell 4 → Cell 5**

### Lần thứ 2 trở đi (skip extract)

1. Sửa `config.yaml` ở local → push lên GitHub
2. Colab: **Cell 1** (pull code mới, ~10s)
3. **BỎ QUA Cell 2 + Cell 3** (data đã có)
4. Sửa `EXP_NAME` + `HYPOTHESIS` ở Cell 4 *(exp10 đã điền sẵn)*
5. **Cell 3.5 → Cell 4 → Cell 5**

### Sau khi Colab xong

6. Local: `git pull` để lấy file MD về
7. Điền 3 section vào file MD: Observations / Conclusion / Next steps
8. `git push`

> Lần đầu setup Cell 5: tạo file `/content/drive/MyDrive/Forget-MI-Project/.git-secrets.json` (xem hướng dẫn trong cell).

In [13]:
# ====================================
# CELL 1: Kết nối Drive & Pull Code (giữ data, không clone lại)
# ====================================
from google.colab import drive
import os

# 1. Mount Google Drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive', force_remount=True)
else:
    print("✅ Google Drive đã được kết nối!")

# 2. Pull code mới (KHÔNG xóa thư mục → data đã extract được giữ nguyên)
%cd /content
REPO = "Forget-MI-LoKU"
REPO_URL = "https://github.com/nhnhu146/Forget-MI-LoKU.git"

if not os.path.exists(REPO):
    print(f"🔽 Clone lần đầu: {REPO}")
    !git clone {REPO_URL}
else:
    print(f"🔄 Pull code mới (giữ data đã extract)")
    %cd {REPO}
    !git fetch origin
    !git reset --hard origin/master 2>&1 | tail -3
    %cd /content

%cd {REPO}
!git log --oneline -1

# 3. Cài đặt thư viện (chỉ chạy lần đầu hoặc khi cần update)
import importlib.util
need_install = importlib.util.find_spec("peft") is None or importlib.util.find_spec("pydicom") is None
if need_install:
    print("📦 Cài đặt thư viện...")
    !pip install -q pydicom scikit-image wandb pyyaml pandas
    !pip install -q "transformers==4.38.0" "peft==0.10.0" "accelerate==0.27.0"
else:
    print("✅ Thư viện đã cài, bỏ qua.")

print("\n✅ Môi trường và mã nguồn đã sẵn sàng!")
print("ℹ️  Lần đầu: chạy Cell 2 → Cell 3 (extract data).")
print("ℹ️  Lần sau: bỏ qua Cell 2 + Cell 3, đi thẳng Cell 3.5 → Cell 4 → Cell 5.")

In [14]:
# ====================================
# CELL 2: Giải nén Data & Models
# ====================================
!python setup_data.py

In [15]:
# ====================================
# CELL 3: Tiền xử lý & Thiết lập Output
# ====================================
# CHỈ CHẠY LẦN ĐẦU (sau đó cache features được giữ trong /content/.../data/metadata/)
import os
import shutil

# 1. Tạo all_data.tsv từ các file báo cáo (idempotent)
!python make_tsv.py

# 2. Cache features — KHÔNG xóa nữa (xóa = phải regenerate 5+ phút mỗi lần)
#    Uncomment 2 dòng dưới nếu bạn THỰC SỰ muốn force regenerate cache
# !rm -f ./data/metadata/cachedfeatures_train_seqlen-*
# !rm -f ./data/metadata/cachednoisyfeatures_train_seqlen-*

# 3. Kết nối thư mục Output với Drive để lưu bền vững
DRIVE_RESULTS = "/content/drive/MyDrive/Forget-MI-Project/unlearning_output"
os.makedirs(DRIVE_RESULTS, exist_ok=True)

if os.path.exists("unlearning_output"):
    if os.path.islink("unlearning_output"):
        os.unlink("unlearning_output")
    else:
        shutil.rmtree("unlearning_output")

!ln -s "{DRIVE_RESULTS}" ./unlearning_output

# 4. Verify cache files có sẵn không
cache_dir = "./data/metadata"
has_cache = False
if os.path.exists(cache_dir):
    files = os.listdir(cache_dir)
    has_cache = any(f.startswith(("cachedfeatures_train_seqlen", "cachednoisyfeatures_train_seqlen"))
                    for f in files)

print(f"\n✅ Output sẽ được lưu tại: {DRIVE_RESULTS}")
print(f"{'✅' if has_cache else '⚠️ '} Cache features {'đã có' if has_cache else 'CHƯA có'} trong {cache_dir}")
if not has_cache:
    print("   → Cell 4 lần đầu sẽ chậm (~5 phút regenerate features). Lần sau sẽ load cache nhanh.")

In [ ]:
# ====================================
# CELL 3.5: Verify config — confirm code mới nhất từ GitHub
# ====================================
# Chạy cell này TRƯỚC Cell 4 để chắc chắn config đúng với exp đang định chạy.
# Nếu thấy giá trị CŨ → bạn quên push từ local, hãy push rồi rerun Cell 1.

print("📋 Config hiện tại (các tham số hay đổi giữa các exp):\n")
!grep -E "^\s*(forget_margin|eta_re_anchor|alpha|beta|theta|gamma|lora_r|lora_alpha|lora_target_modules|use_noise|unlearn_epochs|learning_rate|kappa_cls_retain|kappa_cls_forget|cls_forget_clamp|unfreeze_classifier_heads|uniform_prior_weight|distill_retain_weight|distill_forget_weight|distill_temperature|loku_subtract_scale|ihl_forget_weight|lora_image_last_k_blocks|lora_image_include_fc1|loku_image_subtract_scale):" -A 1 config.yaml | grep -v "^--"

print("\n🧪 Kiểm tra code có fix Exp 03+ và exp10 image-FILA:")
!grep -c "Unfroze classifier\|L_cls_ret\|L_cls_frg\|L_distill_ret\|L_distill_frg\|L_ihl\|TRUE-SUBTRACTION\|resolve_image_targets\|image_target_names\|_fila_decompose" training/forgetmi_loku.py | xargs -I{} echo "   → Số lần khớp pattern fix: {} (nên >= 12)"

print("\n🖼️  exp10 sanity — image PEFT phải BẬT (last_k >= 1):")
!grep -E "^\s*lora_image_last_k_blocks:" -A 1 config.yaml | grep "value:" | xargs -I{} echo "   → {} (0 = TẮT = quay về exp08)"

print("\n🔍 Git commit đang chạy:")
!git log --oneline -1

print("\n👉 Nếu config KHÔNG đúng với exp bạn định chạy:")
print("   1. Local: kiểm tra `git status` xem đã commit chưa")
print("   2. Local: `git push`")
print("   3. Colab: chạy lại Cell 1 (clone fresh)")
print("   4. Chạy lại Cell 3.5 này để verify")

In [ ]:
# ====================================
# CELL 4: Chạy LoKU Unlearning (với auto-tracking)
# ====================================
# ✅ exp10c đã điền sẵn — KHÔNG cần sửa gì, chỉ bấm chạy.
#    (Lần sau muốn chạy exp khác thì mới đổi 2 biến dưới.)
EXP_NAME   = "exp10c_image_fila_distill_forget"
HYPOTHESIS = ("exp10b cho thay scale image-FILA (0.3 vs 0.5) KHONG doi MIA (0.607 vs 0.612) -> "
              "scale khong phai can gat. Chan doan: mo conv anh trainable -> fit retain tot (CLS_ret "
              "1.28->0.78) -> loss forget thap (giong member) -> MIA ~0.61. Gold retrained MIA=0 vi "
              "forget loss cua no ~ test. exp10c giu image-FILA (scale 0.3) + bat distill_forget_weight=1.0 "
              "(nhe) de ep logits forget khop teacher retrained -> keo phan bo forget ve 'chua tung thay' -> "
              "MIA giam. Doi: MIA <= ~0.57, Forget-AUC giu thap ~0.80, Test giu cao.")

# (Optional) Đổi sang True nếu muốn xóa checkpoint cũ trước khi train
FRESH_START = True

# ----- Chạy training với auto-tracking -----
fresh_flag = "--fresh" if FRESH_START else ""
!PYTHONPATH=. WANDB_MODE=disabled python training/forgetmi_loku.py \
    --config config.yaml {fresh_flag} \
    --exp {EXP_NAME} \
    --hypothesis "{HYPOTHESIS}"

print("\n" + "="*60)
print(f"📄 Xem file experiment: experiments/exp_*_{EXP_NAME}.md")
print(f"📊 INDEX (bảng tổng):   experiments/INDEX.md")
print("="*60)
print("👉 Chạy CELL 5 để auto-commit & push lên GitHub")

In [18]:
# ====================================
# CELL 5: Auto-commit & push experiment results lên GitHub
# ====================================
# 3 cách setup credentials (chọn 1, theo độ tiện):
#
# CÁCH A — Lưu vào Drive (KHUYẾN NGHỊ, setup 1 lần dùng mãi):
#   Tạo file /content/drive/MyDrive/Forget-MI-Project/.git-secrets.json
#   với nội dung:
#   {
#     "GITHUB_TOKEN": "ghp_xxxxxxxxxxxx",
#     "GIT_EMAIL": "ban@gmail.com",
#     "GIT_NAME": "Nguyen Hoang Nhu"
#   }
#   Tạo token tại: https://github.com/settings/tokens (scope: repo)
#
# CÁCH B — Colab Secrets (CHỈ web colab.research.google.com):
#   Click 🔑 ở sidebar → Add secret: GITHUB_TOKEN, GIT_EMAIL, GIT_NAME
#
# CÁCH C — Nhập tay mỗi session (lazy, không cần setup):
#   Bỏ qua A và B → cell sẽ tự hỏi token mỗi lần chạy
# ===========================================================
import os, json, getpass
from pathlib import Path

GITHUB_REPO = "nhnhu146/Forget-MI-LoKU"
BRANCH = "master"

def load_secrets():
    # CÁCH A — file trên Drive
    drive_path = Path("/content/drive/MyDrive/Forget-MI-Project/.git-secrets.json")
    if drive_path.exists():
        s = json.loads(drive_path.read_text())
        print(f"🔑 Đã load credentials từ {drive_path}")
        return s.get('GITHUB_TOKEN'), s.get('GIT_EMAIL'), s.get('GIT_NAME')

    # CÁCH B — Colab Secrets (web Colab)
    try:
        from google.colab import userdata
        t = userdata.get('GITHUB_TOKEN')
        if t:
            print("🔑 Đã load credentials từ Colab Secrets")
            return t, userdata.get('GIT_EMAIL'), userdata.get('GIT_NAME')
    except Exception:
        pass

    # CÁCH B2 — environment variables
    if os.environ.get('GITHUB_TOKEN'):
        print("🔑 Đã load credentials từ env vars")
        return (os.environ['GITHUB_TOKEN'],
                os.environ.get('GIT_EMAIL', ''),
                os.environ.get('GIT_NAME', ''))

    # CÁCH C — nhập tay (fallback)
    print("🔑 Nhập credentials thủ công (sẽ ẩn khi gõ token):")
    print("   (lần sau muốn auto, tạo file Drive theo CÁCH A ở comment trên)")
    t = getpass.getpass("  GitHub token (ghp_...): ").strip()
    e = input("  Git email: ").strip()
    n = input("  Git name:  ").strip()
    return t, e, n


TOKEN, EMAIL, NAME = load_secrets()

if TOKEN and EMAIL and NAME:
    # 1. Configure git identity (chỉ trong repo này, không ảnh hưởng global)
    !git config user.email "{EMAIL}"
    !git config user.name "{NAME}"

    # 2. Inject token vào remote URL (chỉ trong session này)
    !git remote set-url origin https://{TOKEN}@github.com/{GITHUB_REPO}.git

    # 3. Pull trước để tránh conflict
    !git pull --rebase origin {BRANCH} 2>&1 | tail -5

    # 4. Stage CHỈ file experiment
    !git add experiments/ 2>/dev/null

    # 5. Hiển thị thay đổi
    changes = !git diff --cached --name-only
    if changes and any(c.strip() for c in changes):
        print("\n📦 Files sẽ commit:")
        for f in changes:
            if f.strip():
                print(f"   - {f}")

        commit_msg = f"exp {EXP_NAME}: auto-tracked results"
        !git commit -m "{commit_msg}"
        !git push origin {BRANCH}

        print(f"\n✅ Đã push lên GitHub")
        print(f"🔗 Xem online: https://github.com/{GITHUB_REPO}/tree/{BRANCH}/experiments")
    else:
        print("ℹ️  Không có file experiment mới để commit.")
else:
    print("⚠️  Thiếu credentials — bỏ qua push. Setup theo CÁCH A/B/C ở comment trên.")